In [ ]:
# If ucimlrepo is missing, install it inside the current notebook kernel.
try:
    from ucimlrepo import fetch_ucirepo
except ModuleNotFoundError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "ucimlrepo"])
    from ucimlrepo import fetch_ucirepo

from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset

# Reproducibility: keeping one seed makes model comparisons much fairer.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Hyperparameters shared by all three homework experiments.
TRAIN_TEST_SPLIT = 0.8
BATCH_SIZE = 16
NUM_EPOCHS = 500
LOG_EVERY = 100
WEIGHT_DECAY = 1e-4
BASELINE_LR = 0.1
BASELINE_NUM_EPOCHS = NUM_EPOCHS  # use 1500 to match the lecture notebook more closely

# Save outputs beside this notebook when running from the repo root, otherwise use the current folder.
NOTEBOOK_FOLDER = Path("Lecture-1") if Path("Lecture-1").is_dir() else Path(".")

# Prefer GPU if it exists. MPS is Apple's GPU backend on newer Macs.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# Load the same UCI Sonar Mines vs Rocks dataset used in the lecture notebook.
# If the ucimlrepo API has a temporary SSL/server issue, fall back to the raw UCI CSV.
try:
    sonar = fetch_ucirepo(id=151)
    X_df = sonar.data.features
    y_series = sonar.data.targets.iloc[:, 0]
    dataset_name = sonar.metadata.get("name", "Connectionist Bench (Sonar, Mines vs. Rocks)")
except Exception as error:
    print(f"ucimlrepo fetch failed: {error}")
    print("Falling back to the raw UCI Sonar CSV URL.")
    sonar_url = "http://archive.ics.uci.edu/ml/machine-learning-databases/undocumented/connectionist-bench/sonar/sonar.all-data"
    raw_df = pd.read_csv(sonar_url, header=None)
    X_df = raw_df.iloc[:, :60]
    X_df.columns = [f"feature_{index:02d}" for index in range(X_df.shape[1])]
    y_series = raw_df.iloc[:, 60]
    dataset_name = "Connectionist Bench (Sonar, Mines vs. Rocks)"

label_to_id = {"R": 0, "M": 1}
id_to_label = {0: "Rock (R)", 1: "Mine / metal (M)"}

X_all = torch.tensor(X_df.to_numpy(), dtype=torch.float32)
y_all = torch.tensor(y_series.map(label_to_id).to_numpy(), dtype=torch.long)

# Reproducible train/test split.
split_generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(X_all), generator=split_generator)
train_size = int(TRAIN_TEST_SPLIT * len(X_all))
train_idx = permutation[:train_size]
test_idx = permutation[train_size:]

X_train_raw = X_all[train_idx]
y_train = y_all[train_idx]
X_test_raw = X_all[test_idx]
y_test = y_all[test_idx]

# New compared with the lecture baseline: standardize features.
# Important detail: mean/std are fitted ONLY on the train split to avoid test-data leakage.
feature_mean = X_train_raw.mean(dim=0, keepdim=True)
feature_std = X_train_raw.std(dim=0, keepdim=True).clamp_min(1e-6)
X_train = (X_train_raw - feature_mean) / feature_std
X_test = (X_test_raw - feature_mean) / feature_std

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

IN_FEATURES = X_train.shape[1]
NUM_CLASSES = len(label_to_id)
criterion = nn.CrossEntropyLoss()


def reset_seed(seed=SEED):
    """Reset random seeds before creating each model, so comparisons are less noisy."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def accuracy_from_logits(logits, targets):
    predictions = logits.argmax(dim=1)
    return (predictions == targets).float().mean().item()


def train_model(model, optimizer, scheduler, train_loader, num_epochs=NUM_EPOCHS, log_every=LOG_EVERY):
    history = {"loss": [], "accuracy": [], "lr": []}

    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_seen = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_size = X_batch.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == y_batch).sum().item()
            total_seen += batch_size

        # New homework item: the LR scheduler updates the learning rate after each epoch.
        # With cosine annealing, the LR starts higher and smoothly becomes smaller.
        scheduler.step()

        epoch_loss = total_loss / total_seen
        epoch_accuracy = total_correct / total_seen
        current_lr = optimizer.param_groups[0]["lr"]

        history["loss"].append(epoch_loss)
        history["accuracy"].append(epoch_accuracy)
        history["lr"].append(current_lr)

        if epoch == 1 or epoch % log_every == 0 or epoch == num_epochs:
            print(
                f"epoch {epoch:03d}/{num_epochs} | "
                f"loss={epoch_loss:.4f} | "
                f"acc={epoch_accuracy * 100:5.1f}% | "
                f"lr={current_lr:.5f}"
            )

    return history


@torch.no_grad()
def evaluate_model(model, data_loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_seen = 0
    all_predictions = []
    all_targets = []

    for X_batch, y_batch in data_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        predictions = logits.argmax(dim=1)

        batch_size = X_batch.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (predictions == y_batch).sum().item()
        total_seen += batch_size
        all_predictions.extend(predictions.cpu().tolist())
        all_targets.extend(y_batch.cpu().tolist())

    return {
        "loss": total_loss / total_seen,
        "accuracy": total_correct / total_seen,
        "predictions": all_predictions,
        "targets": all_targets,
    }


class BaselineSonarFFNN(nn.Module):
    """Lecture 1 baseline: 60 inputs -> 12 hidden sigmoid units -> 2 logits."""

    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(IN_FEATURES, 12),
            nn.Sigmoid(),
            nn.Linear(12, NUM_CLASSES),
        )

    def forward(self, x):
        return self.network(x)


class_balance = (
    pd.Series(y_all.numpy())
    .map(id_to_label)
    .value_counts()
    .rename_axis("class")
    .reset_index(name="count")
)

print(f"Dataset: {dataset_name}")
print(f"Features: {IN_FEATURES} | Classes: {NUM_CLASSES} | Device: {device}")
print(f"Train samples: {len(train_dataset)} | Test samples: {len(test_dataset)}")
display(class_balance)

print("=" * 80)
print("Training lecture baseline: Linear(60, 12) -> Sigmoid -> Linear(12, 2)")
print("=" * 80)

reset_seed(SEED)
baseline_model = BaselineSonarFFNN().to(device)
baseline_optimizer = torch.optim.SGD(baseline_model.parameters(), lr=BASELINE_LR)

# Scheduler kept with gamma=1.0 so the baseline still uses a fixed LR like the lecture notebook.
baseline_scheduler = torch.optim.lr_scheduler.StepLR(
    baseline_optimizer,
    step_size=BASELINE_NUM_EPOCHS,
    gamma=1.0,
)

baseline_history = train_model(
    baseline_model,
    baseline_optimizer,
    baseline_scheduler,
    train_loader,
    num_epochs=BASELINE_NUM_EPOCHS,
    log_every=LOG_EVERY,
)

baseline_train_summary = pd.DataFrame([
    {
        "model": "Lecture baseline sigmoid 12",
        "hidden_sizes": [12],
        "final_train_loss": baseline_history["loss"][-1],
        "final_train_accuracy": baseline_history["accuracy"][-1],
        "final_lr": baseline_history["lr"][-1],
    }
])

display(baseline_train_summary)


# From now on is the important stuff

This is the homework solution section. The lecture baseline was:

`60 inputs → Linear(12) → Sigmoid → Linear(2 logits)`

The baseline is trained in the setup cell before this section. The final chart compares this baseline with the three homework variants.

The experiments below improve it in three ways:

- **GELU instead of Sigmoid**: [`nn.GELU`](https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html) is a smooth non-linearity commonly used in modern neural networks. More activation choices are in the [PyTorch activation docs](https://docs.pytorch.org/docs/stable/nn.html#non-linear-activations).
- **Different hidden sizes and depths**: `hidden_sizes=[16]`, `[32, 16]`, and `[64, 32, 16]` test progressively deeper/wider feed-forward networks.
- **Learning-rate schedule**: [`CosineAnnealingLR`](https://docs.pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.CosineAnnealingLR.html) starts with a larger learning rate and gradually lowers it, which can make training settle into a better solution. See PyTorch's guide on [adjusting learning rates](https://docs.pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate).
- **Dataset context**: [UCI Sonar dataset](https://archive.ics.uci.edu/dataset/151/connectionist+bench+sonar+mines+vs+rocks) and the original [Gorman & Sejnowski paper](https://papers.cnl.salk.edu/PDFs/Analysis%20of%20Hidden%20Units%20in%20a%20Layered%20Network%20Trained%20to%20Classify%20Sonar%20Targets%201988-2996.pdf).

<svg width="760" height="120" viewBox="0 0 760 120" xmlns="http://www.w3.org/2000/svg">
  <style>
    .box { fill: #eef6ff; stroke: #2f6fbd; stroke-width: 2; rx: 12; }
    .gelu { fill: #fff4e6; stroke: #d9822b; stroke-width: 2; rx: 12; }
    .text { font: 15px sans-serif; fill: #1f2937; text-anchor: middle; dominant-baseline: middle; }
    .arrow { stroke: #4b5563; stroke-width: 2; marker-end: url(#arrowhead); }
  </style>
  <defs>
    <marker id="arrowhead" markerWidth="10" markerHeight="7" refX="9" refY="3.5" orient="auto">
      <polygon points="0 0, 10 3.5, 0 7" fill="#4b5563" />
    </marker>
  </defs>
  <rect class="box" x="20" y="35" width="130" height="50" />
  <text class="text" x="85" y="60">60 sonar inputs</text>
  <line class="arrow" x1="150" y1="60" x2="205" y2="60" />
  <rect class="box" x="210" y="35" width="145" height="50" />
  <text class="text" x="282" y="60">Linear layer</text>
  <line class="arrow" x1="355" y1="60" x2="410" y2="60" />
  <rect class="gelu" x="415" y="35" width="115" height="50" />
  <text class="text" x="472" y="60">GELU</text>
  <line class="arrow" x1="530" y1="60" x2="585" y2="60" />
  <rect class="box" x="590" y="35" width="150" height="50" />
  <text class="text" x="665" y="60">2 output logits</text>
  <text x="370" y="105" font="14px sans-serif" fill="#4b5563" text-anchor="middle">Repeat Linear → GELU once, twice, or three times depending on hidden_sizes.</text>
</svg>


In [ ]:
class GeluSonarFFNN(nn.Module):
    """Flexible feed-forward network for Sonar classification."""

    def __init__(self, input_size, hidden_sizes, output_size, dropout=0.05):
        super().__init__()
        layers = []
        current_size = input_size

        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(current_size, hidden_size))

            # New homework item: GELU replaces the baseline Sigmoid activation.
            # GELU is smooth and does not squash every positive value into the tiny 0..1 range.
            layers.append(nn.GELU())

            # Small dropout helps reduce overfitting on this tiny 208-row dataset.
            if dropout > 0:
                layers.append(nn.Dropout(dropout))

            current_size = hidden_size

        layers.append(nn.Linear(current_size, output_size))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


# Three model variants: each one changes both width and depth.
EXPERIMENTS = [
    {"name": "GELU depth=1 width=16", "hidden_sizes": [16], "lr": 1e-2, "dropout": 0.03},
    {"name": "GELU depth=2 widths=32-16", "hidden_sizes": [32, 16], "lr": 1e-2, "dropout": 0.05},
    {"name": "GELU depth=3 widths=64-32-16", "hidden_sizes": [64, 32, 16], "lr": 1e-2, "dropout": 0.08},
]


def build_model_optimizer_scheduler(config):
    reset_seed(SEED)

    model = GeluSonarFFNN(
        input_size=IN_FEATURES,
        hidden_sizes=config["hidden_sizes"],
        output_size=NUM_CLASSES,
        dropout=config["dropout"],
    ).to(device)

    # AdamW is Adam with decoupled weight decay; weight decay lightly discourages huge weights.
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=WEIGHT_DECAY,
    )

    # New homework item: learning-rate scheduler.
    # CosineAnnealingLR lowers LR smoothly from lr to eta_min over NUM_EPOCHS epochs.
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=NUM_EPOCHS,
        eta_min=config["lr"] * 0.05,
    )

    return model, optimizer, scheduler


In [ ]:
trained_models = {}
histories = {}
train_rows = []

for config in EXPERIMENTS:
    print("=" * 80)
    print(f"Training: {config['name']} | hidden_sizes={config['hidden_sizes']}")
    print("=" * 80)

    model, optimizer, scheduler = build_model_optimizer_scheduler(config)
    history = train_model(model, optimizer, scheduler, train_loader)

    trained_models[config["name"]] = model
    histories[config["name"]] = history
    train_rows.append(
        {
            "model": config["name"],
            "hidden_sizes": config["hidden_sizes"],
            "final_train_loss": history["loss"][-1],
            "final_train_accuracy": history["accuracy"][-1],
            "final_lr": history["lr"][-1],
        }
    )

train_summary = pd.DataFrame(train_rows)
display(train_summary)


In [ ]:
baseline_metrics = evaluate_model(baseline_model, test_loader)

test_rows = [
    {
        "model": "Lecture baseline sigmoid 12",
        "hidden_sizes": [12],
        "test_loss": baseline_metrics["loss"],
        "test_accuracy": baseline_metrics["accuracy"],
    }
]

for config in EXPERIMENTS:
    name = config["name"]
    metrics = evaluate_model(trained_models[name], test_loader)

    test_rows.append(
        {
            "model": name,
            "hidden_sizes": config["hidden_sizes"],
            "test_loss": metrics["loss"],
            "test_accuracy": metrics["accuracy"],
        }
    )

results = pd.DataFrame(test_rows).sort_values("test_accuracy", ascending=False).reset_index(drop=True)
display(results)

best_name = results.loc[0, "model"]

if best_name == "Lecture baseline sigmoid 12":
    best_model = baseline_model
    best_config = {
        "name": best_name,
        "hidden_sizes": [12],
        "activation": "Sigmoid",
        "lr": BASELINE_LR,
    }
else:
    best_model = trained_models[best_name]
    best_config = next(config for config in EXPERIMENTS if config["name"] == best_name)

# Homework save step: save the best model's learned weights and enough info to rebuild it later.
# In a serious project, choose the best model with a validation split, then report once on test.
checkpoint_path = NOTEBOOK_FOLDER / "best_sonar_ffnn_comparison.pt"
torch.save(
    {
        "model_state_dict": best_model.state_dict(),
        "config": best_config,
        "feature_mean": feature_mean,
        "feature_std": feature_std,
        "label_to_id": label_to_id,
        "id_to_label": id_to_label,
        "test_results": results.to_dict(orient="records"),
    },
    checkpoint_path,
)

print(f"Best model: {best_name}")
print(f"Saved checkpoint to: {checkpoint_path.resolve()}")

plt.figure(figsize=(11, 4))
plt.bar(results["model"], results["test_accuracy"] * 100)
plt.ylabel("Test accuracy (%)")
plt.title("Lecture baseline vs GELU FFNN homework variants")
plt.xticks(rotation=20, ha="right")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()


In [ ]:
# Extra export cell: no model-class changes are needed.
# PyTorch's easiest native export is a .pt file. TorchScript can be loaded later without redefining
# BaselineSonarFFNN or GeluSonarFFNN, while the .npz is useful for inspecting raw weights/preprocessing.
import json

export_dir = NOTEBOOK_FOLDER / "exports"
export_dir.mkdir(parents=True, exist_ok=True)

best_model_cpu = best_model.to("cpu").eval()
example_input = X_train[:1].to("cpu")

# 1) Class-free PyTorch export: load later with torch.jit.load(...).
torchscript_path = export_dir / "best_sonar_model_torchscript.pt"
traced_model = torch.jit.trace(best_model_cpu, example_input)
traced_model.save(str(torchscript_path))

# 2) NumPy export: stores learned weights plus the train-split normalization values.
# This is not the usual PyTorch reload path, but it is convenient for inspecting arrays.
npz_path = export_dir / "best_sonar_model_weights_and_preprocessing.npz"
npz_arrays = {
    f"param__{name}": tensor.detach().cpu().numpy()
    for name, tensor in best_model_cpu.state_dict().items()
}
npz_arrays["feature_mean"] = feature_mean.squeeze(0).detach().cpu().numpy()
npz_arrays["feature_std"] = feature_std.squeeze(0).detach().cpu().numpy()
npz_arrays["class_names"] = np.array([id_to_label[index] for index in range(NUM_CLASSES)])
np.savez_compressed(npz_path, **npz_arrays)

# 3) Small JSON metadata file describing what was exported.
metadata_path = export_dir / "best_sonar_model_metadata.json"
metadata = {
    "best_model": best_name,
    "best_config": best_config,
    "torchscript_file": torchscript_path.name,
    "npz_file": npz_path.name,
    "label_to_id": label_to_id,
    "id_to_label": id_to_label,
    "input_features": IN_FEATURES,
    "num_classes": NUM_CLASSES,
    "test_results": json.loads(results.to_json(orient="records")),
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Saved export files:")
print(f"- TorchScript model: {torchscript_path.resolve()}")
print(f"- NPZ weights/preprocessing: {npz_path.resolve()}")
print(f"- Metadata: {metadata_path.resolve()}")

# Quick smoke test for the TorchScript export.
loaded_model = torch.jit.load(str(torchscript_path))
with torch.no_grad():
    logits = loaded_model(example_input)
print(f"TorchScript smoke-test logits shape: {tuple(logits.shape)}")
